# 总模型
$$\text{maximize} \quad \text{收益} - (\gamma_{risk} \cdot \text{风险} + \gamma_{trans} \cdot \text{Commision} + \gamma_{hold} \cdot \text{滑点})$$
## $w$含义
算出来的仓位是还没有除以合约乘数的

## $w$变化假设
真实：$w_{t+1} = (1 + r) \cdot (w_t + z_t)$ ——（这是非线性的，会导致模型极难求解）。

代码逻辑：$w_{t+1} \approx w_t + z_t$ ——（模型假设权重在周期之间不发生因价格涨跌导致的漂移，直接传递下去）。
- 第 1 步：时刻 $t$（规划）
  - 状态：你拿着现在的真实持仓 $w_t$（这是起点，绝对真实）。
  - 动作：优化器开始做梦（MPO 计算）。
  - 产出：生成一串计划 $[z_t, z_{t+1}, z_{t+2} \dots]$。
  - 决策：你只取 $z_t$ 拿去下单，把剩下的计划扔进垃圾桶。
- 第 2 步：时刻 $t \rightarrow t+1$（漂移）
  - 状态：你执行了 $z_t$。事件：收盘了，市场涨跌发生了（$r_t$）。
  - 结果：你的持仓被动变成了 $w_{t+1}^{real}$。
  - 注意：这里发生了“你之前担心的事”——真实的 $w_{t+1}^{real}$ 和优化器昨天预测的 $w_{t+1}^{plan}$ 出现了偏差（Drift Error）。
- 第 3 步：时刻 $t+1$（修正）
  - 状态：第二天早上，你看着真实的 $w_{t+1}^{real}$。
  - 动作：优化器重新开始做梦。它不会沿用昨天的旧计划（因为它已经知道昨天的预测有偏差了）。它以最新的、包含漂移的 $w_{t+1}^{real}$ 为起点，重新规划新的一串 $[z_{t+1}', z_{t+2}', \dots]$。
  - 修正：如果昨天的涨幅让某个股票仓位过重了，今天的新计划 $z_{t+1}'$ 会自动包含“卖出更多”的指令来回调它。

## 基本求解约束
1. 预算约束：$$\sum w_{future} + w_{cash} = 1$$
2. 保证金约束：$|w_{future}| \times \text{初始保证金率} \le Limit_{激进}$
   $$Limit_{激进} \le \frac{\text{维持保证金率}}{\text{初始保证金率}}$$

# Return：单期收益变为多期收益
传入的收益率矩阵减去risk free rate（因为现在的现金列没法直接放）

## 预测有不确定性，对未来的收益打折扣
惩罚项
- delta 是在任意一个时间点，对持仓规模（abs(wplus)）进行惩罚：（return rate-delta）*持仓
- gamma 随时间远离，预测越来越不准：衰减因子 = (时间距离) ** (-gamma_decay)

In [ ]:
import cvxpy as cvx
from cvxportfolio.expression import Expression
from .utils import values_in_time, null_checker

__all__ = ['ReturnsForecast', 'MPOReturnsForecast', 'MultipleReturnsForecasts']


class BaseReturnsModel(Expression):
    pass


class ReturnsForecast(BaseReturnsModel):
    """A single return forecast.

    Attributes:
      alpha_data: A dataframe of return estimates.
      delta_data: A confidence interval around the estimates.
      half_life: Number of days for alpha auto-correlation to halve.
    """

    def __init__(self, returns, delta=0., gamma_decay=None, name=None):
        null_checker(returns)
        self.returns = returns
        null_checker(delta)
        self.delta = delta
        self.gamma_decay = gamma_decay
        self.name = name

    def weight_expr(self, t, wplus, z=None, v=None):
        """Returns the estimated alpha.

        Args:
          t: time estimate is made.
          wplus: An expression for holdings.
          tau: time of alpha being estimated.

        Returns:
          An expression for the alpha.
        """
        alpha = cvx.multiply(
            values_in_time(self.returns, t), wplus)
        alpha -= cvx.multiply(
            values_in_time(self.delta, t), cvx.abs(wplus))
        return cvx.sum(alpha)

    def weight_expr_ahead(self, t, tau, wplus):
        """Returns the estimate at time t of alpha at time tau.

        Args:
          t: time estimate is made.
          wplus: An expression for holdings.
          tau: time of alpha being estimated.

        Returns:
          An expression for the alpha.
        """

        alpha = self.weight_expr(t, wplus)
        if tau > t and self.gamma_decay is not None:
            alpha *= (tau - t).days**(-self.gamma_decay)
        return alpha




如果传入的就是是专门为特定未来时间点（例如“从今天看，三天后的预测收益是多少？”）预先计算好的

In [ ]:
class MPOReturnsForecast(BaseReturnsModel):
    """A single alpha estimation.

    Attributes:
      alpha_data: A dict of series of return estimates.
    """

    def __init__(self, alpha_data):
        self.alpha_data = alpha_data

    def weight_expr_ahead(self, t, tau, wplus):
        """Returns the estimate at time t of alpha at time tau.

        Args:
          t: time estimate is made.
          wplus: An expression for holdings.
          tau: time of alpha being estimated.

        Returns:
          An expression for the alpha.
        """
        return self.alpha_data[(t, tau)].values.T * wplus


## 多种惩罚结果加权合并

In [ ]:

class MultipleReturnsForecasts(BaseReturnsModel):
    """A weighted combination of alpha sources.

    Attributes:
      alpha_sources: a list of alpha sources.
      weights: An array of weights for the alpha sources.
    """

    def __init__(self, alpha_sources, weights):
        self.alpha_sources = alpha_sources
        self.weights = weights

    def weight_expr(self, t, wplus, z=None, v=None):
        """Returns the estimated alpha.

        Args:
            t: time estimate is made.
            wplus: An expression for holdings.
            tau: time of alpha being estimated.

        Returns:
          An expression for the alpha.
        """
        alpha = 0
        for idx, source in enumerate(self.alpha_sources):
            alpha += source.weight_expr(t, wplus) * self.weights[idx]
        return alpha

    def weight_expr_ahead(self, t, tau, wplus):
        """Returns the estimate at time t of alpha at time tau.

        Args:
          t: time estimate is made.
          wplus: An expression for holdings.
          tau: time of alpha being estimated.

        Returns:
          An expression for the alpha.
        """
        alpha = 0
        for idx, source in enumerate(self.alpha_sources):
            alpha += source.weight_expr_ahead(t,
                                              tau, wplus) * self.weights[idx]
        return alpha


# Risk
- gamma (风险厌恶系数)：如果 gamma 为 0，则表示完全不考虑风险。
- w_bench (基准权重): 这个参数允许定义一个基准投资组合（benchmark）。如果提供了 w_bench，风险模型计算的将不再是投资组合的总风险，而是相对于基准的主动风险（Active Risk），也称为跟踪误差（Tracking Error）。
- 多期：对近期风险的关注度远高于对远期风险的关注度，每gamma_half_life天后，风险厌恶系数gamma将减半。

In [ ]:
from abc import abstractmethod
import logging

import cvxpy as cvx
import numpy as np
import pandas as pd

from .costs import BaseCost
from .utils import values_in_time
logger = logging.getLogger(__name__)


__all__ = ['FullSigma', 'EmpSigma', 'SqrtSigma', 'WorstCaseRisk','RobustFactorModelSigma', 'RobustSigma',  'FactorModelSigma']


# def locator(obj, t):
#     """Picks last element before t."""
#     try:
#         if isinstance(obj, pd.Panel):
#             return obj.iloc[obj.axes[0].get_loc(t, method='pad')]

#         elif isinstance(obj.index, pd.MultiIndex):
#             prev_t = obj.loc[:t, :].index.values[-1][0]
#         else:
#             prev_t = obj.loc[:t, :].index.values[-1]

#         return obj.loc[prev_t, :]

#     except AttributeError:  # obj not pandas
#         return obj


class BaseRiskModel(BaseCost):

    def __init__(self, **kwargs):
        self.w_bench = kwargs.pop('w_bench', 0.)
        super(BaseRiskModel, self).__init__()
        self.gamma_half_life = kwargs.pop('gamma_half_life', np.inf)

    def weight_expr(self, t, w_plus, z, value):
        self.expression = self._estimate(t, w_plus - self.w_bench, z, value)
        return self.gamma * self.expression, []

    @abstractmethod
    def _estimate(self, t, w_plus, z, value):
        pass

    def weight_expr_ahead(self, t, tau, w_plus, z, value):
        """Estimate risk model at time tau in the future."""
        if self.gamma_half_life == np.inf:
            gamma_multiplier = 1.
        else:
            decay_factor = 2 ** (-1 / self.gamma_half_life)
            # TODO not dependent on days
            gamma_init = decay_factor ** ((tau - t).days)
            gamma_multiplier = gamma_init * \
                (1 - decay_factor) / (1 - decay_factor)

        return gamma_multiplier * self.weight_expr(t, w_plus, z, value)[0], []

    def optimization_log(self, t):
        if self.expression.value:
            return self.expression.value
        else:
            return np.NaN



输入一个现成的

In [ ]:
class FullSigma(BaseRiskModel):
    """Quadratic risk model with full covariance matrix.

    Args:
        Sigma (:obj:`pd.Panel`): Panel of Sigma matrices,
            or single matrix.

    """

    def __init__(self, Sigma, **kwargs):
        self.Sigma = Sigma  # Sigma is either a matrix or a pd.Panel
        try:
            assert(not pd.isnull(Sigma).values.any())
        except AttributeError:
            assert (not pd.isnull(Sigma).any())
        super(FullSigma, self).__init__(**kwargs)

    def _estimate(self, t, wplus, z, value):
        try:
            self.expression = cvx.quad_form(
                wplus, values_in_time(self.Sigma, t))
        except TypeError:
            self.expression = cvx.quad_form(
                wplus, values_in_time(self.Sigma, t).values)
        return self.expression

## 计算方法
假设协方差矩阵可以分解为 $Σ = S * S^T$（半正定），其中 S 是 Σ 的“平方根”矩阵。

风险$ w^T * Σ * w $就可以被等价地写成$ ||S^T * w||²$。

In [ ]:
class SqrtSigma(BaseRiskModel):

    def __init__(self, sigma_sqrt, **kwargs):
        """returns is dataframe, lookback is int"""
        self.sigma_sqrt = sigma_sqrt
        assert(not np.any(pd.isnull(sigma_sqrt)))
        super(SqrtSigma, self).__init__(**kwargs)

    def _estimate(self, t, wplus, z, value):
        # TODO make sure pandas + cvxpy works
        self.expression = cvx.sum_squares(wplus.T * self.sigma_sqrt.values)
        return self.expression

## 协方差矩阵生成方法
### EmpSigma (经验协方差矩阵模型)
- 原理: 这个模型不需要外部提供协方差矩阵。它会在每个时间点，根据过去一段时间的历史收益率数据，动态地实时计算出一个经验协方差矩阵 $ Σ = (1/T) * R^T * R$，其中 R 是历史收益率矩阵。
- 输入:
  - returns: 一个包含资产历史收益率的 DataFrame。
  - lookback: 一个整数，定义了要使用过去多少期的数据来计算协方差。

In [ ]:
class EmpSigma(BaseRiskModel):
    """Empirical Sigma matrix, built looking at *lookback* past returns."""

    def __init__(self, returns, lookback, **kwargs):
        """returns is dataframe, lookback is int"""
        self.returns = returns
        self.lookback = lookback
        assert(not np.any(pd.isnull(returns)))
        super(EmpSigma, self).__init__(**kwargs)

    def _estimate(self, t, wplus, z, value):
        idx = self.returns.index.get_loc(t)
        # TODO make sure pandas + cvxpy works
        R = self.returns.iloc[max(idx - 1 - self.lookback, 0):idx - 1]
        assert (R.shape[0] > 0)
        self.expression = cvx.sum_squares(R.values * wplus) / self.lookback
        return self.expression

### FactorModelSigma (因子模型)
- 原理: 将资产的风险分解为两个部分：
  - 系统性风险: 由少数几个共同的风险因子（如市场、规模、价值、动量等）驱动的风险。
  - 特异性风险: 每个资产自身独有的、与因子无关的风险。公式: $Σ = F * Σ_f * F^T + D$，其中 F 是因子暴露度矩阵，$Σ_f$ 是因子协方差矩阵，D 是特异性风险的对角矩阵。
- 输入:
  - exposures: 资产在各个因子上的暴露度 F。
    - 基本面因子 (Fundamental Factors):
      - 是什么: 比如市值 (Market Cap)、估值 (P/E,P/B)、盈利能力 (ROE)、行业分类 (Industry) 等。
      - 如何获取: 这些暴露就是资产本身的属性数据
    - 宏观经济因子 (Macroeconomic Factors):
      - 是什么: 比如利率、通货膨胀率、GDP增长率等。
      - 如何获取: 通常通过时间序列回归获得每个资产  r_i 对宏观因子序列 f_macro 的系数 β
  - factor_Sigma: 因子之间的协方差矩阵 $Σ_f$。    
  - idiosync: 每个资产的特异性风险（方差）D。

In [ ]:
class FactorModelSigma(BaseRiskModel):

    def __init__(self, exposures, factor_Sigma, idiosync, **kwargs):
        """Each is a pd.Panel (or ) or a vector/matrix"""
        self.exposures = exposures
        assert (not exposures.isnull().values.any())
        self.factor_Sigma = factor_Sigma
        assert (not factor_Sigma.isnull().values.any())
        self.idiosync = idiosync
        assert(not idiosync.isnull().values.any())
        super(FactorModelSigma, self).__init__(**kwargs)

    def _estimate(self, t, wplus, z, value):
        self.expression = cvx.sum_squares(cvx.multiply(
            np.sqrt(values_in_time(self.idiosync, t)), wplus)) + \
            cvx.quad_form((wplus.T * values_in_time(self.exposures, t).values.T).T,
                          values_in_time(self.factor_Sigma, t).values)
        return self.expression

### RobustSigma (鲁棒协方差模型)
- 原理: 这个模型解决了 FullSigma 的一个痛点：我们预测的协方差矩阵 Σ 本身可能是不准确的。RobustSigma 在标准的$ w^T * Σ * w$ 风险之上，额外增加了一个惩罚项，这个惩罚项与 Σ 预测的不确定性有关。
- 输入:
  - Sigma: 基础的协方差矩阵 Σ。
  - epsilon: 一个控制“鲁棒性”强度的系数。epsilon 越大，对 Σ 不确定性的惩罚就越重。

In [ ]:
class RobustSigma(BaseRiskModel):
    """Implements covariance forecast error risk."""

    def __init__(self, Sigma, epsilon, **kwargs):
        self.Sigma = Sigma  # pd.Panel or matrix
        self.epsilon = epsilon  # pd.Series or scalar
        super(RobustSigma, self).__init__(**kwargs)

    def _estimate(self, t, wplus, z, value):
        self.expression = cvx.quad_form(wplus, values_in_time(self.Sigma, t)) + \
            values_in_time(self.epsilon, t) * \
            (cvx.abs(wplus).T * np.diag(values_in_time(
                self.Sigma, t)))**2

        return self.expression

### RobustFactorModelSigma (鲁棒因子模型)
FactorModelSigma 和 RobustSigma 的结合体。它首先使用因子模型来估计风险，然后针对因子协方差矩阵 Σ_f 的不确定性增加一个鲁棒惩罚项。


In [ ]:
class RobustFactorModelSigma(BaseRiskModel):
    """Implements covariance forecast error risk."""

    def __init__(self, exposures, factor_Sigma, idiosync, epsilon, **kwargs):
        """Each is a pd.Panel (or ) or a vector/matrix"""
        self.exposures = exposures
        assert (not exposures.isnull().values.any())
        self.factor_Sigma = factor_Sigma
        assert (not factor_Sigma.isnull().values.any())
        self.idiosync = idiosync
        assert(not idiosync.isnull().values.any())
        self.epsilon = epsilon
        super(RobustFactorModelSigma, self).__init__(**kwargs)

    def _estimate(self, t, wplus, z, value):
        F = values_in_time(self.exposures, t)
        f = (wplus.T * F.T).T
        Sigma_F = values_in_time(self.factor_Sigma, t)
        D = values_in_time(self.idiosync, t)
        self.expression = cvx.sum_squares(
            cvx.multiply(np.sqrt(D), wplus)) + \
            cvx.quad_form(f, Sigma_F) + \
            self.epsilon * (cvx.abs(f).T * np.sqrt(np.diag(Sigma_F)))**2

        return self.expression

## 多种风险模型结合：WorstCaseRisk (最差情况风险模型)
- 原理: 这个模型本身不计算风险，而是一个“风险元模型”。你可以给它提供一个包含多种不同风险模型的列表（例如，一个 EmpSigma 和一个 FactorModelSigma）。在优化时，它会分别计算每种模型下的投资组合风险，然后取其中最大的那个作为最终的风险值。
- 输入:
riskmodels: 一个包含其他风险模型实例的列表。

In [ ]:
class WorstCaseRisk(BaseRiskModel):

    def __init__(self, riskmodels, **kwargs):
        self.riskmodels = riskmodels
        super(WorstCaseRisk, self).__init__(**kwargs)

    def _estimate(self, t, wplus, z, value):
        self.risks = [risk.weight_expr(t, wplus, z, value)
                      for risk in self.riskmodels]
        return cvx.max_elemwise(*self.risks)

    def optimization_log(self, t):
        """Return data to log in the result object."""
        return pd.Series(index=[model.__class__.__name__ for
                                model in self.riskmodels],
                         data=[risk.value[0, 0] for risk in self.risks])

# Cost
$$\text{Total Cost} = \underbrace{\text{Commission}(z)}_{\text{确定性佣金}} + \underbrace{\text{Slippage}(z)}_{\text{市场摩擦}} + \underbrace{\text{Holding}(w)}_{\text{资金占用}}$$
每种cost模型在总cost的重要程度是不一样的，gamma表示重要程度

In [ ]:

import cvxpy as cvx
import numpy as np
import copy
from .expression import Expression
from .utils import null_checker, values_in_time


__all__ = ['CNFuturesCommission', 'CNFuturesSlippage','NFuturesMarginCost']


class BaseCost(Expression):

    def __init__(self):
        self.gamma = 1.  # it is changed by gamma * BaseCost()

    def weight_expr(self, t, w_plus, z, value):
        cost, constr = self._estimate(t, w_plus, z, value)
        return self.gamma * cost, constr

    def weight_expr_ahead(self, t, tau, w_plus, z, value):
        cost, constr = self._estimate_ahead(t, tau, w_plus, z, value)
        return self.gamma * cost, constr

    def __mul__(self, other):
        """Read the gamma parameter as a multiplication."""
        newobj = copy.copy(self)
        newobj.gamma *= other
        return newobj

    def __rmul__(self, other):
        """Read the gamma parameter as a multiplication."""
        return self.__mul__(other)


## Tcost: 交易成本
1. $z$ 是权重，怎么换算成真实的钱？
- 定义：在模型中，$z_i$ 代表交易金额占总资产的比例。
  - 例如：账户总值 100万，$z_{rb} = 0.05$ $\Rightarrow$ 买入 5万元的螺纹钢。
- 按金额收费 (Price/Ratio)：费用 = 交易金额 $\times$ 费率在模型里 = $|z| \times \text{Ratio}$
- 按手收费 (Volume/Fixed)：费用 = 手数 $\times$ 每手固定费用手数 = 交易金额 / (价格 $\times$ 合约乘数) = $(|z| \cdot V_{total}) / (P \cdot M)$在模型里 (归一化后) = $|z| \times \frac{\text{FixedFee}}{P \cdot M}$
2. 基础费率定义 (Per Asset)
- 开仓费率：$$R_{\text{open}}(t) = \text{Ratio}_{\text{open}} + \frac{\text{Fixed}_{\text{open}}}{P_t \cdot M}$$
- 平昨费率（对应 CSV 的 close）：$$R_{\text{yesterday}}(t) = \text{Ratio}_{\text{close}} + \frac{\text{Fixed}_{\text{close}}}{P_t \cdot M}$$
- 平今费率（对应 CSV 的 close_today）：$$R_{\text{today}}(t) = \text{Ratio}_{\text{today}} + \frac{\text{Fixed}_{\text{today}}}{P_t \cdot M}$$
3. 开平分离：
- 优化器 (Estimate): 使用 (Open + CloseYesterday)/2 进行凸近似。
  - 因为无法判断，且只有j和lh的open和平昨是不一样的
- 模拟器 (Value): 能够根据交易方向和持仓状态，精确区分 Open 和 Close。
    (注：默认优先使用 CloseYesterday 费率，符合'优先平老仓'原则)

面孔 A：_estimate（给优化器看）
角色：军事参谋。

任务：在交易发生前，预测这次交易大概要花多少钱。

特点：必须是凸函数 (Convex)。为了数学上的好算，它不得不做一些近似（比如我们刚才讨论的 (Open+Close)/2）。

谁调用它：SinglePeriodOpt（策略对象）。策略用它来决定“怎么下单最划算”。

面孔 B：value_expr（给模拟器看）
角色：铁面会计。

任务：在交易发生后，计算这次交易真实花了多少钱。

特点：必须是精确数值。它可以包含任何复杂的非凸逻辑（比如“先平昨再平今”、“如果交易量超过100手收惩罚费”等 if-else 逻辑）。

谁调用它：MarketSimulator（回测对象）。模拟器用它来扣除你账户里的现金。

In [ ]:
class CNFuturesCommission(BaseCost):
    """
    中国期货佣金模型 (硬性费用)
    
    [功能特性]
    1. 混合计费：自动识别并计算 '按金额(Ratio)' 和 '按手(Fixed)' 两种收费模式。
    2. 动态转化：利用当日价格将 '每手固定费用' 转化为 '等效金额费率'。
    3. 开平分离：
       - 优化器 (Estimate): 使用 (Open + CloseYesterday)/2 进行凸近似。
       - 模拟器 (Value): 能够根据交易方向和持仓状态，精确区分 Open 和 Close。
         (注：默认优先使用 CloseYesterday 费率，符合'优先平老仓'原则)
    """

    def __init__(self, commission_csv_path, multipliers, prices):
        """
        :param commission_csv_path: 包含 open/close/close_today 费率的 CSV 文件路径
        :param multipliers: dict 或 pd.Series, 合约乘数 {ticker: multiplier}
        :param prices: pd.DataFrame, 历史价格数据 (index为时间, columns为ticker)
        """
        super(CNFuturesCommission, self).__init__()
        self.prices = prices
        self.multipliers = multipliers
        self._parse_commission(commission_csv_path)

    def _parse_commission(self, path):
        """解析 CSV 并建立费率映射表"""
        df = pd.read_csv(path)
        self.fee_map = {}
        for _, row in df.iterrows():
            ticker = row['instrument_id']
            self.fee_map[ticker] = {
                'open_ratio': row['open_ratio_by_money'],
                'open_fixed': row['open_fee_by_volume'],
                'close_ratio': row['close_ratio_by_money'], # 平昨
                'close_fixed': row['close_fee_by_volume'],
                # 'today_ratio': row['close_today_ratio_by_money'], # 暂存，供日内模型扩展
                # 'today_fixed': row['close_today_fee_by_volume']
            }

    def _get_dynamic_rate(self, t, asset, rate_type='open'):
        """
        计算 t 时刻的单资产综合费率
        Rate = Ratio + Fixed / (Price * Multiplier)
        """
        fees = self.fee_map.get(asset, None)
        if not fees:
            return 0.0

        # 1. 基础按金额费率
        rate = fees[f'{rate_type}_ratio']
        
        # 2. 按手收费转化
        fixed_fee = fees[f'{rate_type}_fixed']
        if fixed_fee > 0:
            try:
                # 获取 t 时刻价格，处理数据缺失
                P = values_in_time(self.prices[asset], t)
                M = self.multipliers.get(asset, 1.0)
                if P > 0:
                    rate += fixed_fee / (P * M)
            except KeyError:
                pass # 如果取不到价格，忽略固定费部分或报错
        
        return rate

    def _estimate(self, t, w_plus, z, value):
        """
        [优化器接口]
        假设：无法预知 z 是开是平。
        策略：Cost = |z| * (OpenRate + CloseYesterdayRate) / 2
        """
        assets = self.prices.columns
        coeffs = []

        for asset in assets:
            if asset == 'USDOLLAR':
                coeffs.append(0.0)
                continue
            
            # 计算 Open 和 Close(Yesterday) 的动态费率
            r_open = self._get_dynamic_rate(t, asset, 'open')
            r_close = self._get_dynamic_rate(t, asset, 'close')
            
            # 取平均值
            coeffs.append((r_open + r_close) / 2)

        # 构建 cvxpy 参数 (Constant Parameter)
        c_vec = cvx.Parameter(len(coeffs), value=np.array(coeffs), nonneg=True)
        
        return c_vec.T @ cvx.abs(z), []

    def value_expr(self, t, h_plus, u):
        """
        [模拟器接口] 精确扣费
        u: 交易金额向量 (+买入, -卖出)
        h_plus: 交易后持仓金额
        """
        cost = 0.0
        # 确保 u 是 numpy 数组
        u_val = u.values if hasattr(u, 'values') else u
        # h_plus 也是 numpy 数组
        h_val = h_plus.values if hasattr(h_plus, 'values') else h_plus
        
        assets = self.prices.columns

        for i, asset in enumerate(assets):
            trade_amt = u_val[i]
            if abs(trade_amt) < 1e-6 or asset == 'USDOLLAR':
                continue

            # 推算交易前持仓 (h_prev) 以判断开平
            # h_plus = h_prev + u  =>  h_prev = h_plus - u
            h_curr = h_val[i]
            h_prev = h_curr - trade_amt
            
            # 判断逻辑：
            # 1. 如果 h_prev == 0: 显然是开仓 (Open)
            # 2. 如果 h_prev 和 trade_amt 同号: 加仓 (Open)
            # 3. 如果 h_prev 和 trade_amt 异号: 平仓 (Close)
            
            is_open = False
            if abs(h_prev) < 1e-6:
                is_open = True
            elif (h_prev > 0 and trade_amt > 0) or (h_prev < 0 and trade_amt < 0):
                is_open = True
            
            # 费率选择
            if is_open:
                rate = self._get_dynamic_rate(t, asset, 'open')
            else:
                # 按照“优先平昨”原则，默认使用 Close Yesterday 费率
                # 这里暂不处理 intraday 的 Close Today 惩罚，除非引入外部状态
                rate = self._get_dynamic_rate(t, asset, 'close')
                
            cost += abs(trade_amt) * rate
            
        return cost

实例化

In [ ]:
# # 1. 实例化你的自定义成本模型（双面人）
# # 这个对象里既包含了估算逻辑(_estimate)，也包含了核算逻辑(value_expr)
# my_commission = CNFuturesCommission(
#     commission_csv_path='commision.csv', 
#     multipliers=multipliers, 
#     prices=prices
# )

# my_slippage = CNFuturesSlippage(...)
# my_margin = CNFuturesMarginCost(...)

# # 打包成列表
# all_my_costs = [my_commission, my_slippage, my_margin]

# # 2. 传给策略 (Policy) -> 这里会用到 _estimate
# # 告诉优化器：你要按照这些成本去规划交易，别瞎买。
# policy = SinglePeriodOpt(
#     return_forecast=..., 
#     costs=all_my_costs,  # <--- 插入这里
#     constraints=...
# )

# # 3. 传给模拟器 (Simulator) -> 这里会用到 value_expr
# # 告诉模拟器：你要按照这些规则来真实扣费。
# simulator = MarketSimulator(
#     market_returns=...,
#     costs=all_my_costs   # <--- 插入这里 (必须是同一个对象或逻辑一致的对象)
# )

# # 4. 开始回测
# backtest = Backtest(policy, simulator, ...)
# backtest.run()

## Slip: 滑点
1. 方案 A：固定跳数模型 (The Fixed Tick Model)
$$Rate_{slip} = \frac{k \times \text{TickSize}}{P_t}$$
- $k$：跳数系数。$k=1$：假设你每次都滑 1 个点成交（最常见配置）。$k=0.5$：假设你有一半概率滑 1 个点，一半概率不滑（或者你在盘口中间排队）。
- $\text{TickSize}$：品种的物理属性。螺纹钢 = 1元，股指(IF) = 0.2元，国债(T) = 0.005元。
- $P_t$：当前价格。
2. 方案 B：波动率调节模型 (The Volatility Model)
$$Rate_{slip} = k \times \sigma_t$$
- $\sigma_t$：当日（或当时的）预期波动率。
- $k$：调节系数。通常需要通过历史数据拟合，比如取 $0.1$ 到 $0.5$。
3. 方案 C：市场冲击模型 (The Market Impact Model)
   $$Cost_{term} = c \cdot |z|^{1.5}$$
- 为什么是 1.5 次方？
- 成本金额 $\approx$ 交易额 $|z|$ $\times$ 冲击率 $\sqrt{|z|}$
- $|z| \cdot |z|^{0.5} = |z|^{1.5}$

In [ ]:
import cvxpy as cvx
import numpy as np
import pandas as pd
from .expression import Expression
from .utils import values_in_time

class CNFuturesSlippage(BaseCost):
    """
    中国期货通用滑点模型
    
    支持三种计算模式 (通过 config 配置):
    1. 'tick': 固定跳数模型 (Rate = k * Tick / Price) -> 适用于散户/流动性好
    2. 'vol':  波动率模型 (Rate = k * Sigma)         -> 适用于压力测试
    3. 'impact': 市场冲击模型 (Cost ~ |z|^1.5)       -> 适用于机构大资金
    """
    def __init__(self, prices, tick_sizes=None, volumes=None, sigmas=None, 
                 config=None):
        """
        :param prices: 历史价格 DataFrame
        :param tick_sizes: dict {ticker: 0.2, ...}
        :param volumes: 历史成交量 DataFrame (仅 impact 模式需要)
        :param sigmas: 历史波动率 DataFrame (vol 和 impact 模式需要)
        :param config: dict, 例如 {'model': 'tick', 'k': 2.0}
        """
        super(CNFuturesSlippage, self).__init__()
        self.prices = prices
        self.tick_sizes = tick_sizes if tick_sizes else {}
        self.volumes = volumes
        self.sigmas = sigmas
        # 默认配置：滑 1 个 Tick
        self.config = config if config else {'model': 'tick', 'k': 1.0}

    def _estimate(self, t, w_plus, z, value):
        """
        [优化器接口]
        根据配置生成线性或非线性成本表达式
        """
        assets = self.prices.columns
        model = self.config.get('model', 'tick')
        k = self.config.get('k', 1.0)
        
        linear_coeffs = []   # 用于 tick 和 vol 模式
        impact_coeffs = []   # 用于 impact 模式

        for asset in assets:
            if asset == 'USDOLLAR':
                linear_coeffs.append(0.0)
                impact_coeffs.append(0.0)
                continue
            
            rate = 0.0
            impact_c = 0.0
            
            # 数据预取与防错
            try:
                P = values_in_time(self.prices[asset], t)
                # 如果价格无效，给一个极大的惩罚费率防止交易
                if P <= 0: 
                    rate = 1.0 
                else:
                    # --- 模式 A: 固定跳数 ---
                    if model == 'tick':
                        tick = self.tick_sizes.get(asset, 0.0)
                        rate = (k * tick) / P

                    # --- 模式 B: 波动率调节 ---
                    elif model == 'vol':
                        if self.sigmas is not None:
                            sigma = values_in_time(self.sigmas[asset], t)
                            rate = k * sigma

                    # --- 模式 C: 市场冲击 (平方根法则) ---
                    elif model == 'impact':
                        # Cost = Y * sigma * sqrt(TradeVal / (P*V)) * TradeVal
                        # 归一化后系数: Y * sigma * sqrt(PortfolioValue / (P*V))
                        if self.volumes is not None and self.sigmas is not None:
                            V = values_in_time(self.volumes[asset], t)
                            sigma = values_in_time(self.sigmas[asset], t)
                            Y = self.config.get('Y', 1.0) # 冲击系数
                            
                            if V > 0:
                                # 注意：这里 value 是总资产 (Parameter)，在单期优化中视为常数
                                impact_c = Y * sigma * np.sqrt(value / (P * V))
            except KeyError:
                rate = 0.0 # 无数据时不交易

            linear_coeffs.append(rate)
            impact_coeffs.append(impact_c)

        # 构建表达式
        cost_expr = 0
        
        # 1. 线性部分 (Tick / Vol)
        if model in ['tick', 'vol']:
            c_vec = cvx.Parameter(len(linear_coeffs), value=np.array(linear_coeffs), nonneg=True)
            cost_expr = c_vec.T @ cvx.abs(z)
            
        # 2. 非线性部分 (Impact)
        elif model == 'impact':
            c_imp = cvx.Parameter(len(impact_coeffs), value=np.array(impact_coeffs), nonneg=True)
            # 市场冲击通常是 1.5 次方 (Power Law)
            cost_expr = c_imp.T @ cvx.power(cvx.abs(z), 1.5)
            
        return cost_expr, []

    def value_expr(self, t, h_plus, u):
        """
        [模拟器接口] 真实扣费
        """
        u_val = u.values if hasattr(u, 'values') else u
        assets = self.prices.columns
        cost = 0.0
        
        model = self.config.get('model', 'tick')
        k = self.config.get('k', 1.0)
        
        # 估算当前组合价值 (用于 Impact 计算)
        current_val = np.sum(h_plus) 

        for i, asset in enumerate(assets):
            trade_val = abs(u_val[i])
            if trade_val < 1e-6 or asset == 'USDOLLAR': continue
            
            try:
                P = values_in_time(self.prices[asset], t)
                
                if model == 'tick':
                    tick = self.tick_sizes.get(asset, 0.0)
                    cost += trade_val * (k * tick) / P
                    
                elif model == 'vol':
                    sigma = values_in_time(self.sigmas[asset], t)
                    cost += trade_val * k * sigma
                    
                elif model == 'impact':
                    V = values_in_time(self.volumes[asset], t)
                    sigma = values_in_time(self.sigmas[asset], t)
                    Y = self.config.get('Y', 1.0)
                    # 真实冲击成本计算
                    if V > 0 and P > 0:
                        cost += Y * sigma * np.sqrt(trade_val / (P*V)) * trade_val
                        
            except:
                pass
                
        return cost

组合这三种情况：选最差的

但返回给simulator的是模型2

In [ ]:
class WorstCaseSlippage(BaseCost):
    """
    最差情景滑点模型 (Robust Slippage)
    
    逻辑：
    同时计算多个滑点模型，取成本最高的那一个作为优化目标。
    Cost = Max( Cost_Model_1, Cost_Model_2, ... )
    
    用途：
    保守估计交易成本，防止在极端行情下因低估滑点而过度交易。
    """
    def __init__(self, slippage_models):
        """
        :param slippage_models: list, 包含多个 CNFuturesSlippage 实例
        """
        super(WorstCaseSlippage, self).__init__()
        self.models = slippage_models

    def _estimate(self, t, w_plus, z, value):
        """
        [优化器接口]
        返回 cvx.max(cost1, cost2, ...)
        """
        costs = []
        constrs = []
        
        for model in self.models:
            c_expr, c_constr = model._estimate(t, w_plus, z, value)
            costs.append(c_expr)
            constrs += c_constr
            
        # 核心：取最大值 (凸函数)
        return cvx.max(cvx.hstack(costs)), constrs

    def value_expr(self, t, h_plus, u):
        """
        [模拟器接口]
        在回测时，通常我们只关心“真实发生了什么”，而不是“最坏会发生什么”。
        所以这里有两种策略：
        1. (默认) 取第一个模型作为真实世界模型。
        2. (保守) 依然取所有模型中的最大值作为扣费。
        
        这里采用策略 1：假设列表中的第二个模型是 Base Case (真实情况)，
        其他模型只是为了让优化器更谨慎。
        """
        # 使用列表中的第一个模型进行真实扣费
        return self.models[1].value_expr(t, h_plus, u)

## Hcost: 持仓成本
真正被锁定的是initial 保证金: $Cost = sum(|w| * Initial_{Margin}) * R_f$

$R_f$都假设为0.02

In [ ]:
class CNFuturesMarginCost(BaseCost):

    def __init__(self, margin_map, risk_free_rate):
        """
        :param margin_map: dict, 格式 {ticker: {'initial': 0.12, 'maintenance': 0.10}}
        :param risk_free_rate: float, 日化无风险利率 (e.g. 0.0001)
        """
        super(CNFuturesMarginCost, self).__init__()
        self.rf = risk_free_rate
        self.initial_margins = {}
        
        # 严格提取 initial margin
        for ticker, val in margin_map.items():
            if isinstance(val, dict):
                self.initial_margins[ticker] = val.get('initial', 0.1)
            else:
                self.initial_margins[ticker] = float(val)

    def _estimate(self, t, w_plus, z, value):
        # 优化器接口
        assets = self.initial_margins.keys()
        # 注意：实际使用需确保顺序与 w_plus 对应的资产一致
        m_vec_list = [self.initial_margins.get(a, 0.1) for a in assets]
        m_vec = cvx.Parameter(len(m_vec_list), value=np.array(m_vec_list), nonneg=True)
        
        # 资金成本 = 初始保证金占用量 * 无风险利率
        return (m_vec.T @ cvx.abs(w_plus[:-1])) * self.rf, []

    def value_expr(self, t, h_plus, u):
        # 模拟器接口
        cost = 0.0
        # 确保处理 Series 或 Array
        u_val = u.values if hasattr(u, 'values') else u
        h_val = h_plus.values if hasattr(h_plus, 'values') else h_plus
        
        # 假设 assets 顺序与 h_plus 一致（需外部保证）
        assets = self.initial_margins.keys() 
        
        for i, asset in enumerate(assets):
            if asset == 'USDOLLAR': continue
            rate = self.initial_margins.get(asset, 0.1)
            # 无论多空，按初始保证金率冻结资金
            cost += abs(h_val[i]) * rate
            
        return cost * self.rf

# 约束
- 持仓：
  - DollarNeutral(): 美元中性约束：要求投资组合中多头头寸的总价值等于空头头寸的总价值，常用于市场中性策略。
- 交易
  - MaxTrade(ADVs, max_fraction): 最大交易规模限制
  - 对每个资产的交易金额不能超过其历史日均交易量（ADVs）的某个百分比（max_fraction）。
- 因子暴露
  - FactorMaxLimit(factor_exposure, limit) 
  -  factor_exposure.T * w_plus[:-1] <= limit (或 >= limit)。这里 factor_exposure 是一个 n x k 的矩阵（n个资产, k个因子），w_plus[:-1] 是资产权重。矩阵乘法的结果是一个长度为 k 的向量，代表了整个组合在 k 个因子上的净暴露。
- 目标

In [ ]:
from abc import ABCMeta, abstractmethod

import cvxpy as cvx
import numpy as np

from .utils import values_in_time


__all__ = ['LongOnly', 'LeverageLimit', 'LongCash', 'DollarNeutral', 'MaxWeights', 'MinWeights',
        'MaxTrade',
        'FactorMaxLimit', 'FactorMinLimit',
        'FixedAlpha','MarginMaxLeverage']

class BaseConstraint(object):
    __metaclass__ = ABCMeta

    def __init__(self, **kwargs):
        self.w_bench = kwargs.pop('w_bench', 0.)

    def weight_expr(self, t, w_plus, z, v):
        """Returns a list of trade constraints.

        Args:
          t: time
          w_plus: post-trade weights
          z: trade weights
          v: portfolio value
        """
        if w_plus is None:
            return self._weight_expr(t, None, z, v)
        return self._weight_expr(t, w_plus - self.w_bench, z, v)

    @abstractmethod
    def _weight_expr(self, t, w_plus, z, v):
        pass


class MarginMaxLeverage(BaseConstraint):
    """
    [Constraint] 保证金防爆仓约束
    逻辑：总初始保证金占用 <= 账户权益 * limit
    
    limit 设置技巧：
    如果不想要 "维持保证金" 风险，limit 建议设为 0.8 ~ 0.9。
    这相当于预留了 10%~20% 的安全垫来应对 Initial 和 Maintenance 之间的差额。
    """
    def __init__(self, margin_map, limit=0.9):
        self.limit = limit
        self.initial_margins = {}
        for k, v in margin_map.items():
            self.initial_margins[k] = v['initial'] if isinstance(v, dict) else float(v)

    def _weight_expr(self, t, w_plus, z, value):
        # 注意：这里需要严谨处理 asset 顺序对齐，实际使用建议传入 aligned numpy array
        # 下面代码假设 dict values 顺序与 w_plus 对应的资产顺序巧合一致 (慎用!)
        # 正确做法：在策略初始化时，根据 returns.columns 重新排列 margin values
        
        m_vals = np.array(list(self.initial_margins.values()))
        m_vec = cvx.Parameter(len(m_vals), value=m_vals, nonneg=True)
        
        return [m_vec.T @ cvx.abs(w_plus[:-1]) <= self.limit]



持仓限制

In [ ]:
class AssetAbsoluteLimit(BaseConstraint):
    """
    [Constraint] 分品种绝对持仓限制 (Position Limit)
    
    逻辑：|w_i| <= Limit_{i,t}
    含义：无论做多还是做空，单一品种的持仓权重绝对值不能超过限制。
    
    参数 limit_map: 
        - float: 所有品种统一限制 (如 0.2)
        - pd.Series: 分品种限制 (index=资产名)
        - pd.DataFrame: 分品种、分时间动态限制 (index=时间, columns=资产名)
    """
    def __init__(self, limit_map, **kwargs):
        self.limit_map = limit_map
        super(AssetAbsoluteLimit, self).__init__(**kwargs)

    def _weight_expr(self, t, w_plus, z, v):
        # 获取当前时刻的限制 (自动处理 float/Series/DataFrame)
        limits = values_in_time(self.limit_map, t)
        
        # 约束：所有风险资产(排除现金)的绝对权重 <= 限制
        return [cvx.abs(w_plus[:-1]) <= limits]


class TradeAbsoluteLimit(BaseConstraint):
    """
    [Constraint] 分品种绝对开仓/交易限制 (Trading Limit)
    
    逻辑：|z_i| <= Limit_{i,t}
    含义：限制单日交易权重的绝对值。
    注意：在优化器中，较难严格区分'开仓'和'平仓'。这里限制的是'净交易量'。
         既限制了开仓过猛，也限制了平仓过猛（防止冲击成本）。
    
    参数 limit_map: 同上
    """
    def __init__(self, limit_map, **kwargs):
        self.limit_map = limit_map
        super(TradeAbsoluteLimit, self).__init__(**kwargs)

    def _weight_expr(self, t, w_plus, z, v):
        # 获取当前时刻的限制
        limits = values_in_time(self.limit_map, t)
        
        # 约束：所有风险资产的交易权重绝对值 <= 限制
        return [cvx.abs(z[:-1]) <= limits]

# 规划求解（policy）

In [ ]:

from abc import ABCMeta, abstractmethod
import pandas as pd
import numpy as np
import logging
import cvxpy as cvx
from datetime import datetime

from cvxportfolio.costs import BaseCost
from cvxportfolio.returns import BaseReturnsModel
from cvxportfolio.constraints import BaseConstraint
from cvxportfolio.utils import values_in_time, null_checker


__all__ = ['Hold', 'FixedTrade', 'PeriodicRebalance',  
           'SinglePeriodOpt', 'MultiPeriodOpt', 'ProportionalTrade',
           'RankAndLongShort','FuturesRoundTrade']


class BasePolicy(object, metaclass=ABCMeta):
    """ Base class for a trading policy. """

    def __init__(self):
        self.costs = []
        self.constraints = []

    @abstractmethod
    def get_trades(self, portfolio, t=datetime.today()):
        """Trades list given current portfolio and time t.
        """
        return NotImplemented

    def _nulltrade(self, portfolio):
        return pd.Series(index=portfolio.index, data=0.)

    def get_rounded_trades(self, portfolio, prices, t):
        """Get trades vector as number of shares, rounded to integers."""
        return np.round(self.get_trades(portfolio,
                                        t) / values_in_time(prices, t))[:-1]



Hold(): 持有不动策略。最简单的策略，永远不产生任何交易。

SellAll(): 全部卖出策略。产生卖出所有非现金资产的交易指令。

FixedTrade(tradevec, tradeweight): 固定交易策略。每次都执行一个完全相同的、预先定义好的交易。可以按固定金额 (tradevec) 或固定权重 (tradeweight) 来定义。

RankAndLongShort(...): 排名多空策略。一个经典的多空策略（alpha strategy）。

根据你提供的收益预测 (return_forecast) 对所有资产进行排名。
做多排名最高的 num_long 个资产，做空排名最差的 num_short 个资产。
target_turnover 参数控制了每次调仓的力度。

ProportionalTrade(targetweight, time_steps): 比例交易策略。

如果你想在未来 N 个交易日内，把仓位逐渐调整到一个目标权重 (targetweight)，就可以用这个策略。它会在每个交易日平均地完成一部分交易。

PeriodicRebalance(target, period): 周期性再平衡策略。

非常常见的策略。它会在每个固定的时间周期（如 period='month' 每月初，period='year' 每年初）将投资组合的权重严格调整回预设的目标权重 (target)。在两个再平衡日之间则不进行任何交易。


In [ ]:
class Hold(BasePolicy):
    """Hold initial portfolio.
    """

    def get_trades(self, portfolio, t=datetime.today()):
        return self._nulltrade(portfolio)


class RankAndLongShort(BasePolicy):
    """Rank assets, long the best and short the worst (cash neutral)."""

    def __init__(self, return_forecast, num_long, num_short, target_turnover):
        self.target_turnover = target_turnover
        self.num_long = num_long
        self.num_short = num_short
        self.return_forecast = return_forecast
        super(RankAndLongShort, self).__init__()

    def get_trades(self, portfolio, t=datetime.today()):
        prediction = values_in_time(self.return_forecast, t)
        sorted_ret = prediction.sort_values()

        short_trades = sorted_ret.index[:self.num_short]
        long_trades = sorted_ret.index[-self.num_long:]

        u = pd.Series(0., index=prediction.index)
        u[short_trades] = -1.
        u[long_trades] = 1.
        u /= sum(abs(u))
        u *= self.target_leverage # (放大到比如 3 倍)
        u = sum(portfolio) * u * self.target_turnover

        # import pdb; pdb.set_trace()
        #
        # # ex-post cash neutrality
        # old_cash = portfolio[-1]
        # if old_cash > 0:
        #     u[short] = u[short] + old_cash/self.num_short
        # else:
        #     u[long] = u[long] + old_cash/self.num_long

        return u


class ProportionalTrade(BasePolicy):
    """Gets to target in given time steps."""

    def __init__(self, targetweight, time_steps):
        self.targetweight = targetweight
        self.time_steps = time_steps
        super(ProportionalTrade, self).__init__()

    def get_trades(self, portfolio, t=datetime.today()):
        try:
            missing_time_steps = len(
                self.time_steps) - next(i for (i, x)
                                        in enumerate(self.time_steps)
                                        if x == t)
        except StopIteration:
            raise Exception(
                "ProportionalTrade can only trade on the given time steps")
        deviation = self.targetweight - portfolio / sum(portfolio)
        return sum(portfolio) * deviation / missing_time_steps


class SellAll(BasePolicy):
    """Sell all non-cash assets."""

    def get_trades(self, portfolio, t=datetime.today()):
        trade = -pd.Series(portfolio, copy=True)
        trade.ix[-1] = 0.
        return trade


class FixedTrade(BasePolicy):
    """Trade a fixed trade vector.
    """

    def __init__(self, tradevec=None, tradeweight=None):
        """Trade the tradevec vector (dollars) or tradeweight weights."""
        if tradevec is not None and tradeweight is not None:
            raise Exception
        if tradevec is None and tradeweight is None:
            raise Exception
        self.tradevec = tradevec
        self.tradeweight = tradeweight
        assert(self.tradevec is None or sum(self.tradevec) == 0.)
        assert(self.tradeweight is None or sum(self.tradeweight) == 0.)
        super(FixedTrade, self).__init__()

    def get_trades(self, portfolio, t=datetime.today()):
        if self.tradevec is not None:
            return self.tradevec
        return sum(portfolio) * self.tradeweight


class BaseRebalance(BasePolicy):

    def _rebalance(self, portfolio):
        return sum(portfolio) * self.target - portfolio


class PeriodicRebalance(BaseRebalance):
    """Track a target portfolio, rebalancing at given times.
    """

    def __init__(self, target, period, **kwargs):
        """
        Args:
            target: target weights, n+1 vector
            period: supported options are "day", "week", "month", "quarter",
                "year".
                rebalance on the first day of each new period
        """
        self.target = target
        self.period = period
        super(PeriodicRebalance, self).__init__()

    def is_start_period(self, t):
        result = not getattr(t, self.period) == getattr(self.last_t,
                                                        self.period) if \
            hasattr(self,
                    'last_t')\
            else True
        self.last_t = t
        return result

    def get_trades(self, portfolio, t=datetime.today()):
        return self._rebalance(portfolio) if self.is_start_period(t) else \
            self._nulltrade(portfolio)


## SPO

In [ ]:
import cvxpy as cvx
import pandas as pd
import numpy as np
import logging
from datetime import datetime
from .policies import BasePolicy
from .costs import BaseCost
from .constraints import BaseConstraint
from .utils import null_checker, values_in_time

class SinglePeriodOpt(BasePolicy):
    """
    单周期优化器 (Single Period Optimization)
    核心逻辑：Maximize(Alpha - Costs) s.t. Constraints
    """
    def __init__(self, return_forecast, costs, constraints, solver=None, solver_opts=None):
        
        # 1. 收益模型检查
        if not hasattr(return_forecast, 'weight_expr'):
            # 如果不是类模型，那就是静态数据，做个简单的 null check
            # (注意：此处保留原逻辑，但建议 return_forecast 最好是对象)
            null_checker(return_forecast)
            
        self.return_forecast = return_forecast
        
        # 2. 成本和约束检查
        self.costs = []
        for cost in costs:
            assert isinstance(cost, BaseCost)
            self.costs.append(cost)

        self.constraints = []
        for constraint in constraints:
            assert isinstance(constraint, BaseConstraint)
            self.constraints.append(constraint)

        # 3. 求解器配置 (建议默认使用 OSQP 或 CLARABEL 以提高稳定性)
        self.solver = solver if solver else 'OSQP'
        self.solver_opts = solver_opts if solver_opts else {'verbose': False}

        super(SinglePeriodOpt, self).__init__()

    def get_trades(self, portfolio, t=None):
        """
        计算最优交易向量 (单位: 美元/元)
        """
        if t is None:
            t = datetime.now() # 使用 now() 比 today() 更精确

        # 1. 准备数据
        value = sum(portfolio)
        w = portfolio / value
        
        # 定义优化变量 z (权重变化量)
        z = cvx.Variable(w.size) 
        wplus = w.values + z

        # 2. 构建目标函数 (Alpha Term)
        if hasattr(self.return_forecast, 'weight_expr'):
            # 使用模型对象计算
            alpha_term = self.return_forecast.weight_expr(t, wplus, z, value)
        else:
            # 使用静态数据计算
            alpha_term = cvx.sum(cvx.multiply(
                values_in_time(self.return_forecast, t).values, 
                wplus
            ))

        assert(alpha_term.is_concave())

        # 3. 构建成本项 (Costs)
        costs_exprs = []
        constraints = [cvx.sum(z) == 0] # 基础约束：权重变化和为0 (Budget Constraint)

        for cost in self.costs:
            # 传入 value 以支持金额相关的成本计算
            c_expr, c_constr = cost.weight_expr(t, wplus, z, value)
            costs_exprs.append(c_expr)
            constraints += c_constr

        # 4. 构建约束项 (Constraints)
        for con in self.constraints:
            constraints += con.weight_expr(t, wplus, z, value)

        # 5. 定义并求解问题
        # Maximize Alpha - Sum(Costs)
        obj = cvx.Maximize(alpha_term - sum(costs_exprs))
        self.prob = cvx.Problem(obj, constraints)

        try:
            self.prob.solve(solver=self.solver, **self.solver_opts)

            # 6. 处理求解结果
            if self.prob.status in ['unbounded', 'infeasible']:
                logging.error(f'Optimization failed: {self.prob.status}. No trades.')
                return pd.Series(0., index=portfolio.index)

            # 返回交易金额向量 (Dollars)
            return pd.Series(index=portfolio.index, data=(z.value * value))

        except cvx.SolverError as e:
            logging.error(f'Solver {self.solver} failed: {e}')
            return pd.Series(0., index=portfolio.index)

## 输出结果转化为手

In [ ]:
import pandas as pd
import os

def load_multipliers(path):
    """
    读取合约乘数 CSV 文件并转换为字典
    """
    if not os.path.exists(path):
        raise FileNotFoundError(f"找不到乘数文件: {path}")
    
    # 假设 csv 包含 'instrument_id' 和 'multiplier' 两列
    # 如果列名不同，请根据实际情况修改 header
    df = pd.read_csv(path)
    
    # 清洗数据：去除空格，转为字典 { 'RB2310': 10, 'IF2309': 300 }
    multipliers = dict(zip(
        df['instrument_id'].str.strip(), 
        df['multiplier']
    ))
    
    return multipliers

# 使用示例
# multiplier_path = r"C:\Users\CtaBackup\Desktop\opt\data\contract_multipliers.csv"
# multipliers_dict = load_multipliers(multiplier_path)



In [ ]:
class FuturesRoundTrade(BasePolicy):
    """
    [Wrapper Policy] 期货手数取整策略
    
    功能：
    1. 接收 SPO 算出的连续交易金额 (Trade Dollars)。
    2. 读取合约乘数表 (Multiplier)。
    3. 计算目标手数 = (当前持仓 + 计划交易) / (价格 * 乘数)。
    4. 执行取整 (默认为向下取整 floor)。
    5. 逆算真实交易金额，并用现金 (Cash) 补齐差额。
    """
    def __init__(self, base_policy, multipliers_path, prices, round_mode='floor'):
        """
        :param base_policy: 你的 SPO 策略实例
        :param multipliers_path: CSV 文件路径 (字符串)
        :param prices: 历史价格 DataFrame
        :param round_mode: 'floor' (向下取整, 推荐), 'round' (四舍五入)
        """
        self.base_policy = base_policy
        self.prices = prices
        self.round_mode = round_mode
        
        # 在初始化时加载乘数表
        self.multipliers = load_multipliers(multipliers_path)

    def _recursive_values_in_time(self, t, **kwargs):
        # 传递递归调用，确保 base_policy 能获取数据
        if hasattr(self.base_policy, '_recursive_values_in_time'):
            self.base_policy._recursive_values_in_time(t, **kwargs)

    def get_trades(self, portfolio, t=None):
        # 1. 获取 base_policy (SPO) 的原始交易建议 (金额)
        raw_trades = self.base_policy.get_trades(portfolio, t)
        
        # 如果没有交易信号，直接返回
        if raw_trades.abs().sum() < 1e-4:
            return raw_trades

        # 2. 准备取整容器
        rounded_trades = pd.Series(0.0, index=portfolio.index)
        portfolio_value = sum(portfolio)
        
        # 3. 遍历每个资产进行取整 (跳过现金)
        for asset in portfolio.index:
            if asset == 'USDOLLAR': 
                continue # 现金最后统一配平
            
            # 获取乘数 (如果没有配置，默认乘数为1或者跳过)
            multiplier = self.multipliers.get(asset)
            if multiplier is None:
                # 如果不在表中（如股票ETF等），可能不需要乘数，或者直接用原始交易
                # 这里假设不在表中的不做处理，直接用原始值（或报错，取决于你需求）
                # logging.warning(f"{asset} multiplier not found, skipping round.")
                rounded_trades[asset] = raw_trades[asset]
                continue

            # 获取当前价格
            try:
                P = values_in_time(self.prices[asset], t)
            except KeyError:
                # 缺价格数据，无法交易
                rounded_trades[asset] = 0.0
                continue
                
            if P <= 0: continue

            # --- 核心取整逻辑 ---
            
            # A. 算出目标持仓金额 (Target Dollars)
            current_dollars = portfolio[asset]
            trade_dollars = raw_trades[asset]
            target_dollars = current_dollars + trade_dollars
            
            # B. 算出目标手数 (Target Hands)
            # 公式: 金额 / (价格 * 乘数)
            target_hands_float = target_dollars / (P * multiplier)
            
            # C. 执行取整
            if self.round_mode == 'floor':
                # 向下取整 (保持符号): 1.8 -> 1, -1.8 -> -1
                target_hands_int = int(abs(target_hands_float)) * np.sign(target_hands_float)
            else:
                # 四舍五入
                target_hands_int = int(round(target_hands_float))
                
            # D. 逆算真实持仓金额
            real_target_dollars = target_hands_int * P * multiplier
            
            # E. 算出修正后的交易金额
            rounded_trades[asset] = real_target_dollars - current_dollars

        # 4. 配平现金 (Budget Balance)
        # 所有的取整误差（买不起碎股剩下的钱）都回到现金账户
        # sum(trades) 必须为 0
        non_cash_trades_sum = rounded_trades.drop('USDOLLAR', errors='ignore').sum()
        rounded_trades['USDOLLAR'] = -non_cash_trades_sum
        
        return rounded_trades

实例化

In [ ]:
# # 1. 路径定义
# csv_path = r"C:\Users\CtaBackup\Desktop\opt\data\contract_multipliers.csv"

# # 2. 定义内部 SPO 策略
# spo_policy = SinglePeriodOpt(
#     return_forecast=return_model,
#     costs=[commission_model, slippage_model, hcost_model],
#     constraints=[leverage_limit],
#     solver='OSQP' # 推荐
# )

# # 3. 套上取整外壳 (Wrapper)
# # 最终传给回测器的是这个 final_policy
# final_policy = FuturesRoundTrade(
#     base_policy=spo_policy,
#     multipliers_path=csv_path,  # 传入你的 CSV 路径
#     prices=prices_df,           # 传入全历史价格表
#     round_mode='floor'          # 建议用 floor 比较安全
# )

# # 4. 运行回测
# backtest = Backtest(final_policy, simulator, ...)

## MPO

In [ ]:
class MultiPeriodOpt(SinglePeriodOpt):

    def __init__(self, trading_times,
                 terminal_weights, lookahead_periods=None, *args, **kwargs):
        """
        trading_times: list, 所有的交易时间点
        lookahead_periods: int or None. 向前看几步
        """
        self.lookahead_periods = lookahead_periods
        self.trading_times = trading_times
        self.terminal_weights = terminal_weights
        super(MultiPeriodOpt, self).__init__(*args, **kwargs)

    def get_trades(self, portfolio, t=datetime.today()):

        value = sum(portfolio)
        assert (value > 0.)
        # 1. 当前权重 (Constant)
        w = cvx.Constant(portfolio.values / value)

        # 收集所有的目标函数项和约束项
        total_obj_expr = 0
        total_constraints = []
        
        # 用于记录第一步的交易变量 (我们需要返回它)
        first_z = None

        # 确定时间窗口
        try:
            start_idx = self.trading_times.index(t)
        except ValueError:
             # 如果时间对不上，做个容错
            return self._nulltrade(portfolio)
            
        end_idx = start_idx + self.lookahead_periods if self.lookahead_periods else len(self.trading_times)
        planning_times = self.trading_times[start_idx:end_idx]

        # --- 你的原版循环逻辑 (完全保留) ---
        for i, tau in enumerate(planning_times):
            
            # 每一新步，定义一个新的 z 变量
            z = cvx.Variable(*w.shape)
            if i == 0:
                first_z = z # 记录第一步
            
            # 链式传导：这一步的期末 = 上一步的期末 + 这一步的交易
            wplus = w + z
            
            # A. 收益项
            # (注意：如果你的 return_forecast 没有 weight_expr_ahead 方法，这里会报错，
            #  如果报错，请告诉我，我给你加个简单的补丁，这里先按你原代码写)
            obj = self.return_forecast.weight_expr_ahead(t, tau, wplus)

            costs, constr = [], []
            for cost in self.costs:
                # B. 成本项
                cost_expr, const_expr = cost.weight_expr_ahead(t, tau, wplus, z, value)
                costs.append(cost_expr)
                constr += const_expr

            obj -= sum(costs)
            
            # C. 预算约束 (直接写在这里了)
            constr += [cvx.sum(z) == 0]
            
            # D. 其他约束 (如保证金)
            constr += [con.weight_expr(t, wplus, z, value)
                       for con in self.constraints]

            # --- 修改点：不再创建 prob_arr，而是累加到总表达式里 ---
            # 原代码: prob_arr.append(cvx.Problem(...)) -> sum(prob_arr)
            # 修改后:
            total_obj_expr += obj
            total_constraints += constr
            
            # 关键：更新 w，把这一步的期末作为下一步的起点
            w = wplus

        # Terminal constraint (期末约束)
        if self.terminal_weights is not None:
            total_constraints += [wplus == self.terminal_weights.values]

        # --- 求解 ---
        # 构造一个包含所有时间步的大问题
        prob = cvx.Problem(cvx.Maximize(total_obj_expr), total_constraints)
        
        # 使用你指定的 solver
        prob.solve(solver=self.solver, **self.solver_opts)

        # 返回第一步的 z
        return pd.Series(index=portfolio.index, data=(first_z.value * value))

# simulator

In [ ]:

import copy
import logging
import time

import multiprocess
import numpy as np
import pandas as pd
import cvxpy as cvx

from .returns import MultipleReturnsForecasts

from .result import SimulationResult
from .costs import BaseCost

# TODO update benchmark weights (?)
# Also could try jitting with numba.


class MarketSimulator():
    logger = None

    def __init__(self, market_returns, costs,
                 market_volumes=None, cash_key='cash'):
        """Provide market returns object and cost objects."""
        self.market_returns = market_returns
        if market_volumes is not None:
            self.market_volumes = market_volumes[
                market_volumes.columns.difference([cash_key])]
        else:
            self.market_volumes = None

        self.costs = costs
        for cost in self.costs:
            assert (isinstance(cost, BaseCost))

        self.cash_key = cash_key

    def propagate(self, h, u, t):
        """Propagates the portfolio forward over time period t, given trades u.

        Args:
            h: pandas Series object describing current portfolio
            u: n vector with the stock trades (not cash)
            t: current time

        Returns:
            h_next: portfolio after returns propagation
            u: trades vector with simulated cash balance
        """
        assert (u.index.equals(h.index))

        if self.market_volumes is not None:
            # don't trade if volume is null
            null_trades = self.market_volumes.columns[
                self.market_volumes.loc[t] == 0]
            if len(null_trades):
                logging.info('No trade condition for stocks %s on %s' %
                             (null_trades, t))
                u.loc[null_trades] = 0.

        hplus = h + u
        costs = [cost.value_expr(t, h_plus=hplus, u=u) for cost in self.costs]
        for cost in costs:
            assert(not pd.isnull(cost))
            assert(not np.isinf(cost))

        u[self.cash_key] = - sum(u[u.index != self.cash_key]) - sum(costs)
        hplus[self.cash_key] = h[self.cash_key] + u[self.cash_key]

        assert (hplus.index.sort_values().equals(
            self.market_returns.columns.sort_values()))
        h_next = self.market_returns.loc[t] * hplus + hplus

        assert (not h_next.isnull().values.any())
        assert (not u.isnull().values.any())
        return h_next, u

    def run_backtest(self, initial_portfolio, start_time, end_time,
                     policy, loglevel=logging.WARNING):
        """Backtest a single policy.
        """
        logging.basicConfig(level=loglevel)

        results = SimulationResult(initial_portfolio=copy.copy(
            initial_portfolio),
            policy=policy, cash_key=self.cash_key,
            simulator=self)
        h = initial_portfolio

        simulation_times = self.market_returns.index[
            (self.market_returns.index >= start_time) &
            (self.market_returns.index <= end_time)]
        logging.info('Backtest started, from %s to %s' %
                     (simulation_times[0], simulation_times[-1]))

        for t in simulation_times:
            logging.info('Getting trades at time %s' % t)
            start = time.time()
            try:
                u = policy.get_trades(h, t)
            except cvx.SolverError:
                logging.warning(
                    'Solver failed on timestamp %s. Default to no trades.' % t)
                u = pd.Series(index=h.index, data=0.)
            end = time.time()
            assert (not pd.isnull(u).any())
            results.log_policy(t, end - start)

            logging.info('Propagating portfolio at time %s' % t)
            start = time.time()
            h, u = self.propagate(h, u, t)
            end = time.time()
            assert (not h.isnull().values.any())
            results.log_simulation(t=t, u=u, h_next=h,
                                   risk_free_return=self.market_returns.loc[
                                       t, self.cash_key],
                                   exec_time=end - start)

        logging.info('Backtest ended, from %s to %s' %
                     (simulation_times[0], simulation_times[-1]))
        return results

    def run_multiple_backtest(self, initial_portf, start_time,
                              end_time, policies,
                              loglevel=logging.WARNING, parallel=True):
        """Backtest multiple policies.
        """

        def _run_backtest(policy):
            return self.run_backtest(initial_portf, start_time, end_time,
                                     policy, loglevel=loglevel)

        num_workers = min(multiprocess.cpu_count(), len(policies))
        if parallel:
            workers = multiprocess.Pool(num_workers)
            results = workers.map(_run_backtest, policies)
            workers.close()
            return results
        else:
            return list(map(_run_backtest, policies))

    def what_if(self, time, results, alt_policies, parallel=True):
        """Run alternative policies starting from given time.
        """
        # TODO fix
        initial_portf = copy.copy(results.h.loc[time])
        all_times = results.h.index
        alt_results = self.run_multiple_backtest(initial_portf,
                                                 time,
                                                 all_times[-1],
                                                 alt_policies, parallel)
        for idx, alt_result in enumerate(alt_results):
            alt_result.h.loc[time] = results.h.loc[time]
            alt_result.h.sort_index(axis=0, inplace=True)
        return alt_results

    @staticmethod
    def reduce_signal_perturb(initial_weights, delta):
        """Compute matrix of perturbed weights given initial weights."""
        perturb_weights_matrix = \
            np.zeros((len(initial_weights), len(initial_weights)))
        for i in range(len(initial_weights)):
            perturb_weights_matrix[i, :] = initial_weights / \
                (1 - delta * initial_weights[i])
            perturb_weights_matrix[i, i] = (1 - delta) * initial_weights[i]
        return perturb_weights_matrix

    def attribute(self, true_results, policy,
                  selector=None,
                  delta=1,
                  fit="linear",
                  parallel=True):
        """Attributes returns over a period to individual alpha sources.

        Args:
            true_results: observed results.
            policy: the policy that achieved the returns.
                    Alpha model must be a stream.
            selector: A map from SimulationResult to time series.
            delta: the fractional deviation.
            fit: the type of fit to perform.
        Returns:
            A dict of alpha source to return series.
        """
        # Default selector looks at profits.
        if selector is None:
            def selector(result):
                return result.v - sum(result.initial_portfolio)

        alpha_stream = policy.return_forecast
        assert isinstance(alpha_stream, MultipleReturnsForecasts)
        times = true_results.h.index
        weights = alpha_stream.weights
        assert np.sum(weights) == 1
        alpha_sources = alpha_stream.alpha_sources
        num_sources = len(alpha_sources)
        Wmat = self.reduce_signal_perturb(weights, delta)
        perturb_pols = []
        for idx in range(len(alpha_sources)):
            new_pol = copy.copy(policy)
            new_pol.return_forecast = MultipleReturnsForecasts(alpha_sources,
                                                               Wmat[idx, :])
            perturb_pols.append(new_pol)
        # Simulate
        p0 = true_results.initial_portfolio
        alt_results = self.run_multiple_backtest(p0, times[0], times[-1],
                                                 perturb_pols, parallel)
        # Attribute.
        true_arr = selector(true_results).values
        attr_times = selector(true_results).index
        Rmat = np.zeros((num_sources, len(attr_times)))
        for idx, result in enumerate(alt_results):
            Rmat[idx, :] = selector(result).values
        Pmat = cvx.Variable((num_sources, len(attr_times)))
        if fit == "linear":
            prob = cvx.Problem(cvx.Minimize(0), [Wmat * Pmat == Rmat])
            prob.solve()
        elif fit == "least-squares":
            error = cvx.sum_squares(Wmat * Pmat - Rmat)
            prob = cvx.Problem(cvx.Minimize(error),
                               [Pmat.T * weights == true_arr])
            prob.solve()
        else:
            raise Exception("Unknown fitting method.")
        # Dict of results.
        wmask = np.tile(weights[:, np.newaxis], (1, len(attr_times))).T
        data = pd.DataFrame(columns=[s.name for s in alpha_sources],
                            index=attr_times,
                            data=Pmat.value.T * wmask)
        data['residual'] = true_arr - np.matrix((weights * Pmat).value).A1
        data['RMS error'] = np.matrix(
            cvx.norm(Wmat * Pmat - Rmat, 2, axis=0).value).A1
        data['RMS error'] /= np.sqrt(num_sources)
        return data


class CNFuturesSimulator(MarketSimulator):
    """
    中国期货专用模拟器
    
    核心改进：
    1. 强平逻辑 (Margin Call Logic)：权益不足时强制平仓。
    2. 修正现金流逻辑：不应该对'名义杠杆产生的负现金'收利息。
    """
    def __init__(self, trading_times, costs, margin_map, 
                 maintenance_margin_buffer=0.8, **kwargs):
        """
        :param margin_map: 保证金字典 {ticker: ratio} (用于计算强平线)
        :param maintenance_margin_buffer: 强平阈值。
               如果 (占用保证金 > 总权益 * buffer)，触发强平。
               通常维持保证金是初始的 0.7~0.8 倍。
        """
        super(CNFuturesSimulator, self).__init__(trading_times, costs, **kwargs)
        self.margin_map = {}
        for k, v in margin_map.items():
            self.margin_map[k] = v['initial'] if isinstance(v, dict) else float(v)
            
        self.limit_threshold = maintenance_margin_buffer

    def propagate(self, t, u, h):
        """
        覆写状态更新逻辑
        t: 当前时间
        u: 交易向量 (Trades in dollars)
        h: 持仓向量 (Holdings in dollars, 昨天的)
        """
        # 1. 先调用父类逻辑，算出单纯的净值变化 (h_plus = h + u)
        #    以及扣除成本后的下期持仓 (h_next)
        h_next = super(CNFuturesSimulator, self).propagate(t, u, h)
        
        # 如果父类返回 None (如破产)，直接传递
        if h_next is None:
            return None

        # 2. --- 新增：强平检查 (Margin Call Check) ---
        
        # 计算当前总权益 (Equity)
        equity = h_next.sum()
        
        # 如果已经穿仓（负资产），直接宣告死亡
        if equity <= 0:
            logging.warning(f"[{t}] BANKRUPTCY! Equity: {equity:.2f}")
            # 返回全 0 或者 None 表示破产
            return pd.Series(0., index=h_next.index)

        # 计算当前持仓占用的初始保证金 (Initial Margin Used)
        # 注意：模拟器里的 h 是名义价值，所以可以直接乘保证金率
        margin_used = 0.0
        for asset, value in h_next.items():
            if asset == 'USDOLLAR': continue
            rate = self.margin_map.get(asset, 0.1) # 默认 10%
            margin_used += abs(value) * rate
            
        # 检查是否击穿维持保证金线
        # 逻辑：如果 占用保证金 > 权益 (说明杠杆过高，风险度 > 100%)
        # 这里用 limit_threshold (比如 1.0 或 1.1) 来控制容忍度
        # 通常交易所看的是：风险度 = 维持保证金 / 权益 > 100%
        
        # 简化版强平：如果 初始保证金占用 > 权益 * 1.2 (假设维持是初始的80%左右)
        # 或者更严格：Margin_Initial > Equity (这时候其实还没爆，但已经很危险了)
        
        # 我们用一个宽松的强平线：权益跌到连维持保证金都不够了
        # 维持保证金 ≈ 0.8 * 初始保证金
        maintenance_margin = margin_used * 0.8 
        
        if maintenance_margin > equity:
            logging.warning(f"[{t}] MARGIN CALL TRIGGERED! Equity: {equity:.2f}, MaintMargin: {maintenance_margin:.2f}")
            
            # 执行强平：强制将所有持仓砍半，或者直接清零
            # 这里简单处理：直接清仓 (Close All)
            h_liquidated = pd.Series(0., index=h_next.index)
            h_liquidated['USDOLLAR'] = equity # 剩下的钱都是现金
            return h_liquidated

        return h_next